# Trade Analysis Solution (Imports/Exports)

Worked solution aligned to `trade_analysis_exercise.ipynb`.


In [ ]:
import os
from pathlib import Path
import urllib.request as request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.colors import TwoSlopeNorm


In [ ]:
# Set display options for better readability of dataframes
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## 0) Load data and standardize columns


In [ ]:

trade_data_path = Path('../../../data/0_raw/malawi/trade data/Trade Data Python Training.xlsx')

df = pd.read_excel(trade_data_path)

# requested standardization right after loading
df.columns = df.columns.str.lower()

df.head()

## 1) Cleaning and replacements


In [ ]:
d = df.copy()
# Build usdvalue from FX table only when missing
if 'usdvalue' not in d.columns or d['usdvalue'].isna().all():
    fx = pd.DataFrame({
        'year': [2024] * 12,
        'period': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
        'mwk_per_usd': [1700, 1710, 1720, 1730, 1740,
                        1750, 1760, 1770, 1780, 1790, 1800, 1810]
    })

    d = d.merge(fx, on=['year', 'period'], how='left', validate='m:1')
    d['usdvalue'] = d['mwkvalue'] / d['mwk_per_usd']
    d = d.drop(columns=['mwk_per_usd'])

# Ensure numeric dtype either way
d['usdvalue'] = pd.to_numeric(d['usdvalue'], errors='coerce')

d[['year', 'period', 'mwkvalue', 'usdvalue']].head()


In [ ]:


d['partner'] = d['partner'].astype(str).str.strip().str.upper()
d['flow'] = d['flow'].astype(str).str.strip().str.upper()

d['hscode'] = d['hscode'].astype('Int64').astype(str).str.replace('<NA>', '', regex=False)

if 'hscode2' not in d.columns:
    d['hscode2'] = pd.to_numeric(d['hscode'].str[:2], errors='coerce')
else:
    d['hscode2'] = pd.to_numeric(d['hscode2'], errors='coerce')

for col in ['year', 'period', 'mwkvalue', 'usdvalue']:
    if col in d.columns:
        d[col] = pd.to_numeric(d[col], errors='coerce')

d[['year', 'period', 'flow', 'partner', 'hscode', 'hscode2', 'mwkvalue', 'usdvalue']].head()


## 2) Trade-Value Features and Outlier Handling


In [ ]:
#- `flow_sign`: map `I` and `R` to `-1`, and `E` and `RE` to `+1`
sign_map = {'I': -1, 'R': -1, 'E': 1, 'RE': 1}
d['flow_sign'] = d['flow'].map(sign_map)

In [ ]:
# calculate `value_band`: create 5 classes based on `usdvalue` we can use either `pd.qcut` 
labels5 = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
d['value_band'] = pd.qcut(d['usdvalue'], q=5, labels=labels5)

In [ ]:
#or `pd.cut` with quantiles as bins. The two solutions are equivalent, but the second one is more flexible if you want to use custom quantiles or add more classes.
quantiles = d['usdvalue'].quantile([0, 0.2, 0.4, 0.6, 0.8, 1])
d['value_band'] = pd.cut(d['usdvalue'], quantiles, labels=labels5)

In [ ]:
#Plot a boxplot of `usdvalue`

In [ ]:
# We can use the IQR method to identify outliers in `usdvalue` and filter them out for a cleaner analysis. 
q1 = d['usdvalue'].quantile(0.25)
q3 = d['usdvalue'].quantile(0.75)
iqr = q3 - q1
lower = max(0, q1 - 1.5 * iqr)
upper = q3 + 1.5 * iqr

d_no_outliers = d[d['usdvalue'].between(lower, upper, inclusive='both')].copy()
d_no_outliers[['flow', 'flow_sign', 'usdvalue', 'value_band']].head()

In [ ]:
# Once we have the filtered dataset, we can compare the distribution of `usdvalue` 
# before and after cleaning using boxplots to visualize the effect of outlier removal.
# Boxplot comparison: before vs after cleaning
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].boxplot(d['usdvalue'].dropna(), vert=False)
axes[0].set_title('Before cleaning (usdvalue)')
axes[0].set_xlabel('USD value')

axes[1].boxplot(d_no_outliers['usdvalue'].dropna(), vert=False)
axes[1].set_title('After cleaning (usdvalue)')
axes[1].set_xlabel('USD value')
plt.tight_layout()

print('Rows before:', len(d))
print('Rows after outlier filter:', len(d_no_outliers))
print('Filtered out:', len(d) - len(d_no_outliers))

## 3) Create a date field


In [ ]:
# To facilitate time-based analysis, we can create a `date` column by combining the `year` and `period` columns. 
# As `period` represents months, we can construct a date using the first day of each month for simplicity. 
# We can also extract the month name for easier interpretation in visualizations.
d_work = d_no_outliers.copy()

d_work['date'] = pd.to_datetime(
    d_work['year'].astype(str)
    + '-'
    + d_work['period'].astype(str)
    + '-01',
    errors='coerce'
)
d_work['month_name'] = d_work['date'].dt.month_name()

d_work[['year', 'period', 'date', 'month_name']].head()


## 4) Seasonality from month name


In [ ]:
# We can also create a `flow_group` column to categorize the flows into 'Import/Reimport' and 'Export/Reexport' based on the `flow_sign` values.
d_work['flow_group'] = d_work['flow_sign'].map({-1: 'Import/Reimport', 1: 'Export/Reexport'})

seasonality = (
    d_work.groupby(['period', 'month_name', 'flow_group'])['mwkvalue']
    .sum()
    .unstack(fill_value=0)
    .sort_index(level='period')
)
seasonality.head()

In [ ]:
# and finally we can plot the seasonality by month name and flow group to visualize any patterns in imports and exports throughout the year.
seasonality.plot(kind='line', figsize=(12, 4), rot=45)
plt.title('Seasonality by month name and flow group')
plt.xlabel('Month')
plt.ylabel('Total MWK')
plt.tight_layout()

## 5) Join + scatter visualization


In [ ]:
# To analyze trade by partner, we can group the data by `partner` and `flow_sign` to calculate total imports and exports in both MWK and USD. 
# This will allow us to identify key trading partners and their contribution to overall trade.
partner_imports = (
    d[d['flow_sign']==-1]
    .groupby('partner', dropna=False)
    .agg(imports_mwk=('mwkvalue', 'sum'), imports_usd=('usdvalue', 'sum'))
    .reset_index()
)

partner_exports = (
    d[d['flow_sign']==1]
    .groupby('partner', dropna=False)
    .agg(exports_mwk=('mwkvalue', 'sum'), exports_usd=('usdvalue', 'sum'))
    .reset_index()
)

print('Partner imports:')
print(partner_imports.head())
print('Partner exports:')
partner_exports.head()

In [ ]:
# We can then merge the imports and exports dataframes on `partner` to get a complete picture of trade with each partner, including the trade balance in both MWK and USD.

# Please note that we use `how='outer'` to include all partners, even those that only have imports or exports, 
# and we fill missing values with 0 to avoid issues in balance calculations.
partner_trade = partner_exports.merge(partner_imports, on='partner', how='outer').fillna(0)
partner_trade['balance_mwk'] = partner_trade['exports_mwk'] - partner_trade['imports_mwk']
partner_trade['balance_usd'] = partner_trade['exports_usd'] - partner_trade['imports_usd']
partner_trade['balance_sign'] = np.where(partner_trade['balance_mwk'] >= 0, 'Surplus', 'Deficit')

# Finally, we can create a scatter plot of imports vs exports in MWK to visualize the trade balance with each partner. 
x = partner_trade['imports_mwk']
y = partner_trade['exports_mwk']

partner_trade.head()

In [ ]:
# In the scatter plot, we can use different colors to indicate whether a partner has a trade surplus or deficit, 
# and we can set both axes to a logarithmic scale to better visualize the distribution of trade values, especially if there are large disparities between partners.
fig, ax = plt.subplots(figsize=(8, 6))
colors = partner_trade['balance_sign'].map({'Surplus': '#1f77b4', 'Deficit': '#d62728'})
ax.scatter(x, y, c=colors, alpha=0.7)

max_axis = max(x.max(), y.max())
ax.plot([1, max_axis], [1, max_axis], '--', color='gray', linewidth=1)
# we set log scale to better visualize the distribution of trade values, especially if there are large disparities between partners.
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Imports (MWK, log scale)')
ax.set_ylabel('Exports (MWK, log scale)')
ax.set_title('Partner trade profile: imports vs exports')

from matplotlib.patches import Patch
legend_handles = [
    Patch(color='#1f77b4', label='Surplus'),
    Patch(color='#d62728', label='Deficit'),
]
ax.legend(handles=legend_handles)

plt.tight_layout()

## 6.1) Geopandas step 1: download/read world country shapes


In [ ]:
# We download the world shapefile from Natural Earth to visualize the trade data geographically.
world_url = 'https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip'
RAW_CANDIDATES = [
    Path('data/0_raw'),
    Path('../data/0_raw'),
    Path('../../data/0_raw'),
    Path('../../../data/0_raw'),
]
RAW_DATA_DIR = next((p for p in RAW_CANDIDATES if p.exists()), Path('../../../data/0_raw'))
world_zip = RAW_DATA_DIR / 'ne_110m_admin_0_countries.zip'

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
if not world_zip.exists():
    request.urlretrieve(world_url, world_zip)

world = gpd.read_file(world_zip)
world.head()

## 6.2) Geopandas step 2: harmonize names and merge country-level trade balance


In [ ]:
# We can then merge the `partner_trade` dataframe with the `world` geodataframe to create a geospatial dataset 
# that includes trade information for each country.
# We will merge on the `partner` column from `partner_trade` and the `ADMIN` column from `world`, which contains the country names. 
# We will use a left join to keep all partners in the trade data, even if they don't have a corresponding geometry in the world
country_balance = partner_trade[['partner', 'imports_mwk', 'exports_mwk', 'balance_mwk']].copy()
country_balance.head()


In [ ]:
# To ensure a successful merge, we need to standardize the country names in both datasets.
world['partner'] = world['ADMIN'].str.upper().str.strip()
country_balance['partner'] = country_balance['partner'].str.upper().str.strip()

focus = pd.concat([
    country_balance.nlargest(10, 'balance_mwk'),
    country_balance.nsmallest(10, 'balance_mwk')
], axis=0).drop_duplicates(subset=['partner'])
focus_partners = set(focus['partner'])

# We can check which of the focus partners are not matching with the world dataset and create a mapping 
# to fix common discrepancies in country names.
name_fix_candidates = {
    'USA': 'UNITED STATES OF AMERICA',
    'UAE': 'UNITED ARAB EMIRATES',
    'UK': 'UNITED KINGDOM',
    'RUSSIA': 'RUSSIAN FEDERATION',
    'TANZANIA': 'UNITED REPUBLIC OF TANZANIA',
    'DR CONGO': 'DEMOCRATIC REPUBLIC OF THE CONGO'
}
name_fix = {k: v for k, v in name_fix_candidates.items() if k in focus_partners}

country_balance['partner_for_merge'] = country_balance['partner'].replace(name_fix)

In [ ]:
# We can then perform the merge again with the corrected partner 
# names to create the geospatial dataset for visualization.
geo_balance = world.merge(
    country_balance,
    left_on='partner',
    right_on='partner_for_merge',
    how='left'
)

focus_after_fix = focus.copy()
focus_after_fix['partner_for_merge'] = focus_after_fix['partner'].replace(name_fix)
unmatched_focus = sorted(set(focus_after_fix['partner_for_merge']) - set(world['partner']))
print('Unmatched focus countries (top/bottom 10):', unmatched_focus)

## 6.4) Geopandas step 4: map trade balance


In [ ]:
# Finally, we can plot the trade balance geographically using a choropleth map.
geo_balance['balance_mwk'] = geo_balance['balance_mwk'].fillna(0)

absmax = geo_balance['balance_mwk'].abs().max()
norm = TwoSlopeNorm(vmin=-absmax, vcenter=0, vmax=absmax)

ax = geo_balance.plot(
    column='balance_mwk',
    cmap='RdBu',
    norm=norm,
    figsize=(14, 7),
    edgecolor='white',
    linewidth=0.2,
    legend=True,
    missing_kwds={'color': 'lightgrey', 'label': 'No data'}
)
ax.set_title('Trade balance by partner country (MWK)')
ax.set_axis_off()
plt.tight_layout()
